# core

> HTMX v4 support for FastHTML

In [ ]:
#| default_exp core

In [ ]:
#| export
import json

from fastcore.basics import patch
from fastcore.utils import *
from fastcore.xml import *
from fastcore.meta import delegates

from fasthtml.common import *
from fasthtml.starlette import *
from fasthtml.core import *

from fasthtml.core import _wrap_ex, _list, _get_htmx, _fix_anno, _find_wsp, _wrap_ws, _params, _handle, _ws_endp
from fasthtml.fastapp import _get_tbl, _app_factory

In [ ]:
from fasthtml.jupyter import *

In [ ]:
#| export
@delegates(ft_hx)
def Partial(*args, **kwargs): return ft_hx("hx-partial")(*args, **kwargs)


# htmx4 hdrs and metaCharacter

In [ ]:
#| export
htmx4src   = Script(src="https://unpkg.com/htmx.org@4.0.0-alpha6/dist/htmx.js")

In [ ]:
#| export
# When htmx4=True, configures htmx v4 with metaCharacter="-"
def def_hdrs(htmx=True, htmx4=False, surreal=True):
    "Default headers for a FastHTML app"
    hdrs = []
    if surreal: hdrs = [surrsrc,scopesrc] + hdrs
    if htmx and htmx4: raise ValueError("Cannot enable both htmx and htmx4")
    if htmx: hdrs = [htmxsrc,fhjsscr] + hdrs
    if htmx4: 
        # metaCharacter="-" makes htmx4 use dashes instead of colons (Python-friendly)
        meta_cfg = Meta(name="htmx:config", content=json.dumps({"metaCharacter": "-"}))
        hdrs = [meta_cfg, htmx4src,fhjsscr] + hdrs 
    # TODO: Check if fhjsscr works with htmx4
    return [charset, viewport] + hdrs

# FastHTML and fastapp

In [ ]:
# Patch FastHTML.__init__ to add htmx4 support
# - Adds `htmx4` parameter to toggle htmx v4 headers
# - Add self.htmx4
# - Passes htmx4 to def_hdrs() which handles the header selection
# - Maps 'ws' and 'ws4' extensions to 'ws4' when htmx4=True
@patch
def __init__(self: FastHTML, debug=False, routes=None, middleware=None, title: str = "FastHTML page", exception_handlers=None,
                on_startup=None, on_shutdown=None, lifespan=None, hdrs=None, ftrs=None, exts=None,
                before=None, after=None, surreal=True, htmx=True, htmx4=False, default_hdrs=True, sess_cls=SessionMiddleware,
                secret_key=None, session_cookie='session_', max_age=365*24*3600, sess_path='/',
                same_site='lax', sess_https_only=False, sess_domain=None, key_fname='.sesskey',
                body_wrap=noop_body, htmlkw=None, nb_hdrs=False, canonical=True, **bodykw):
    middleware,before,after = map(_list, (middleware,before,after))
    self.title,self.canonical,self.session_cookie,self.key_fname = title,canonical,session_cookie,key_fname
    self.htmx4 = htmx4
    hdrs,ftrs,exts = map(listify, (hdrs,ftrs,exts))
    if htmx4 and exts:
        exts = ['ws4' if e in ('ws', 'ws4') else e for e in exts]
    exts = {k:htmx_exts[k] for k in exts}
    htmlkw = htmlkw or {}
    if default_hdrs: hdrs = def_hdrs(htmx, htmx4, surreal=surreal) + hdrs
    hdrs += [Script(src=ext) for ext in exts.values()]
    if IN_NOTEBOOK:
        hdrs.append(iframe_scr) # TODO: check iframe_scr if work with htmx4
        from IPython.display import display,HTML
        if nb_hdrs: display(HTML(to_xml(tuple(hdrs))))
        middleware.append(cors_allow)
    on_startup,on_shutdown = listify(on_startup) or None,listify(on_shutdown) or None
    self.lifespan,self.hdrs,self.ftrs = lifespan,hdrs,ftrs
    self.body_wrap,self.before,self.after,self.htmlkw,self.bodykw = body_wrap,before,after,htmlkw,bodykw
    self.secret_key = get_key(secret_key, key_fname)
    if sess_cls:
        sess = Middleware(sess_cls, secret_key=self.secret_key,session_cookie=session_cookie,
                            max_age=max_age, path=sess_path, same_site=same_site,
                            https_only=sess_https_only, domain=sess_domain)
        middleware.append(sess)
    exception_handlers = ifnone(exception_handlers, {})
    if 404 not in exception_handlers:
        def _not_found(req, exc): return  Response('404 Not Found', status_code=404)
        exception_handlers[404] = _not_found
    excs = {k:_wrap_ex(v, k, hdrs, ftrs, htmlkw, bodykw, body_wrap=body_wrap) for k,v in exception_handlers.items()}
    super(FastHTML, self).__init__(debug, routes, middleware=middleware, exception_handlers=excs, on_startup=on_startup, on_shutdown=on_shutdown, lifespan=lifespan)

In [ ]:
#| export
# Supports htmx4=True for htmx v4 compatibility
def fast_app(
        db_file:Optional[str]=None, # Database file name, if needed
        render:Optional[callable]=None, # Function used to render default database class
        hdrs:Optional[tuple]=None, # Additional FT elements to add to <HEAD>
        ftrs:Optional[tuple]=None, # Additional FT elements to add to end of <BODY>
        tbls:Optional[dict]=None, # Experimental mapping from DB table names to dict table definitions
        before:Optional[tuple]|Beforeware=None, # Functions to call prior to calling handler
        middleware:Optional[tuple]=None, # Standard Starlette middleware
        live:bool=False, # Enable live reloading
        debug:bool=False, # Passed to Starlette, indicating if debug tracebacks should be returned on errors
        title:str="FastHTML page", # Default page title
        routes:Optional[tuple]=None, # Passed to Starlette
        exception_handlers:Optional[dict]=None, # Passed to Starlette
        on_startup:Optional[callable]=None, # Passed to Starlette
        on_shutdown:Optional[callable]=None, # Passed to Starlette
        lifespan:Optional[callable]=None, # Passed to Starlette
        default_hdrs=True, # Include default FastHTML headers such as HTMX script?
        pico:Optional[bool]=None, # Include PicoCSS header?
        surreal:Optional[bool]=True, # Include surreal.js/scope headers?
        htmx:Optional[bool]=True, # Include HTMX header?
        htmx4:Optional[bool]=False, # Include HTMX4 header?
        exts:Optional[list|str]=None, # HTMX extension names to include
        canonical:bool=True, # Automatically include canonical link?
        secret_key:Optional[str]=None, # Signing key for sessions
        key_fname:str='.sesskey', # Session cookie signing key file name
        session_cookie:str='session_', # Session cookie name
        max_age:int=365*24*3600, # Session cookie expiry time
        sess_path:str='/', # Session cookie path
        same_site:str='lax', # Session cookie same site policy
        sess_https_only:bool=False, # Session cookie HTTPS only?
        sess_domain:Optional[str]=None, # Session cookie domain
        htmlkw:Optional[dict]=None, # Attrs to add to the HTML tag
        bodykw:Optional[dict]=None, # Attrs to add to the Body tag
        reload_attempts:Optional[int]=1, # Number of reload attempts when live reloading
        reload_interval:Optional[int]=1000, # Time between reload attempts in ms
        static_path:str=".",  # Where the static file route points to, defaults to root dir
        body_wrap:callable=noop_body, # FT wrapper for body contents
        nb_hdrs:bool=False, # If in notebook include headers inject headers in notebook DOM?
        **kwargs):
    "Create a FastHTML or FastHTMLWithLiveReload app."
    h = (picolink,) if pico or (pico is None and default_hdrs) else ()
    if hdrs: h += tuple(hdrs)

    app = _app_factory(hdrs=h, ftrs=ftrs, before=before, middleware=middleware, live=live, debug=debug, title=title, routes=routes, exception_handlers=exception_handlers,
                  on_startup=on_startup, on_shutdown=on_shutdown, lifespan=lifespan, default_hdrs=default_hdrs, secret_key=secret_key, canonical=canonical,
                  session_cookie=session_cookie, max_age=max_age, sess_path=sess_path, same_site=same_site, sess_https_only=sess_https_only,
                  sess_domain=sess_domain, key_fname=key_fname, exts=exts, surreal=surreal, htmx=htmx, htmx4=htmx4, htmlkw=htmlkw,
                  reload_attempts=reload_attempts, reload_interval=reload_interval, body_wrap=body_wrap, nb_hdrs=nb_hdrs, **(bodykw or {}))
    app.static_route_exts(static_path=static_path)
    if not db_file: return app,app.route

    db = database(db_file)
    if not tbls: tbls={}
    if kwargs:
        if isinstance(first(kwargs.values()), dict): tbls = kwargs
        else:
            kwargs['render'] = render
            tbls['items'] = kwargs
    dbtbls = [_get_tbl(db.t, k, v) for k,v in tbls.items()]
    if len(dbtbls)==1: dbtbls=dbtbls[0]
    return app,app.route,*dbtbls

# WS

## Summary

Here is the summary of all patches need to be done to make ws work with both htmx v2 and v4
## 1. **Headers & Configuration**
- Created `htmx4src` pointing to htmx v4 alpha
- Modified `def_hdrs()` to add htmx v4 script + meta config with `metaCharacter="-"` (makes attributes use dashes instead of colons)
- Added `ws4` extension URL to `htmx_exts`

## 2. **FastHTML.__init__ Patch**
- Added `htmx4` parameter and stored it as `self.htmx4`
- Auto-maps `'ws'` extension to `'ws4'` when `htmx4=True`

## 3. **WebSocket Parameter Handling** (`_find_wsp_patch`)
- Checks both top-level `data.get(arg)` (htmx v2) AND `data.get('values', {}).get(arg)` (htmx v4)
- Returns `partial(_send_ws4 if htmx4 else _send_ws, ws)` for the `send` parameter

## 4. **Message Sending** (`_send_ws4`)
- Created `WsResponse4` dataclass with `content`, `target`, `swap`, `channel`, `format`
- Wraps HTML in JSON envelope format: `{"channel": "ui", "format": "html", "payload": "...", "target": "#id"}`

## 5. **Threading htmx4 Flag Through Chain**
- `_add_ws` → `_ws_endp(htmx4=self.htmx4)`
- `_ws_endp` → `_wrap_ws(..., htmx4=htmx4)`
- `_wrap_ws` → `_find_wsp(..., htmx4=htmx4)`

## 6. **Multi-Element Returns**
- Created `Partial()` helper for `<hx-partial>` tags
- Allows returning tuples like `(Partial(..., hx_target='#id1'), Partial(..., hx_target='#id2'))`


In [ ]:
from inspect import Parameter
empty = Parameter.empty

In [ ]:
import fasthtml.core as _core
from fasthtml.core import _send_ws

In [ ]:
from typing import Optional, get_type_hints, get_args, get_origin, Union, Mapping, TypedDict, List, Any
from types import UnionType, SimpleNamespace as ns, GenericAlias
from datetime import datetime,date


In [ ]:
from starlette.testclient import TestClient

In [ ]:
htmx_exts

{'morph': 'https://cdn.jsdelivr.net/npm/idiomorph@0.7.3/dist/idiomorph-ext.min.js',
 'head-support': 'https://cdn.jsdelivr.net/npm/htmx-ext-head-support@2.0.4/head-support.js',
 'preload': 'https://cdn.jsdelivr.net/npm/htmx-ext-preload@2.1.1/preload.js',
 'class-tools': 'https://cdn.jsdelivr.net/npm/htmx-ext-class-tools@2.0.1/class-tools.js',
 'loading-states': 'https://cdn.jsdelivr.net/npm/htmx-ext-loading-states@2.0.1/loading-states.js',
 'multi-swap': 'https://cdn.jsdelivr.net/npm/htmx-ext-multi-swap@2.0.0/multi-swap.js',
 'path-deps': 'https://cdn.jsdelivr.net/npm/htmx-ext-path-deps@2.0.0/path-deps.js',
 'remove-me': 'https://cdn.jsdelivr.net/npm/htmx-ext-remove-me@2.0.0/remove-me.js',
 'debug': 'https://unpkg.com/htmx.org@1.9.12/dist/ext/debug.js',
 'ws': 'https://cdn.jsdelivr.net/npm/htmx-ext-ws@2.0.3/ws.js',
 'chunked-transfer': 'https://cdn.jsdelivr.net/npm/htmx-ext-transfer-encoding-chunked@0.4.0/transfer-encoding-chunked.js'}

In [ ]:
htmx_exts['ws4'] = 'https://unpkg.com/htmx.org@4.0.0-alpha6/dist/ext/hx-ws.js'

## WS4 patch

In [ ]:
# NEW: Dataclass for htmx v4 WebSocket responses with JSON envelope fields
@dataclass
class WsResponse4:
    content: Any
    target: str = None
    swap: str = None
    channel: str = "ui"
    format: str = "html"

In [ ]:
# NEW: Sends WS messages as JSON envelope (htmx v4 format) instead of raw HTML
async def _send_ws4(ws, resp):
    "Send WebSocket message with htmx v4 JSON envelope"
    if not resp: return
    if isinstance(resp, WsResponse4):
        payload = to_xml(resp.content)
        msg = dict(channel=resp.channel, format=resp.format, payload=payload, 
                   target=resp.target, swap=resp.swap)
    elif isinstance(resp, dict):
        payload = to_xml(resp.get('content', ''))
        msg = dict(channel=resp.get('channel', 'ui'), format=resp.get('format', 'html'), 
                   payload=payload, target=resp.get('target'), swap=resp.get('swap'))
    else:
        payload = to_xml(resp)
        msg = dict(channel="ui", format="html", payload=payload)
    await ws.send_text(json.dumps(msg))

In [ ]:
# CHANGED: Pass htmx4=self.htmx4 to _ws_endp
@patch
def _add_ws(self:FastHTML, func, path, conn, disconn, name, middleware):
    endp = _ws_endp(func, conn, disconn, htmx4=self.htmx4)
    route = WebSocketRoute(path, endpoint=endp, name=name, middleware=middleware)
    route.methods = ['ws']
    self.add_route(route)
    return func

In [ ]:
# Patch for htmx v4 WebSocket: form fields are now in data['values'] instead of top-level data
def _find_wsp_patch(ws, data, hdrs, arg:str, p:Parameter, htmx4=False):
    "In `data` find param named `arg` of type in `p` (`arg` is ignored for body types)"
    anno = p.annotation
    if isinstance(anno, type):
        if issubclass(anno, HtmxHeaders): return _get_htmx(hdrs)
        if issubclass(anno, Starlette): return ws.scope['app']
        if issubclass(anno, WebSocket): return ws
        if issubclass(anno, dict): return data
    if anno is empty:
        if arg.lower()=='ws': return ws
        if arg.lower()=='scope': return dict2obj(ws.scope)
        if arg.lower()=='data': return data
        if arg.lower()=='htmx': return _get_htmx(hdrs)
        if arg.lower()=='app': return ws.scope['app']
        if arg.lower()=='send': return partial(_send_ws4 if htmx4 else _send_ws, ws)
        if 'session'.startswith(arg.lower()): return ws.scope.get('session', {})
        return None
    res = data.get(arg, None)  # htmx v2: top-level
    if res is empty or res is None: res = data.get('values', {}).get(arg, None)  # htmx v4: in 'values', need to check why we need empty?
    if res is empty or res is None: res = hdrs.get(arg, None)
    if res is empty or res is None: res = p.default
    if not isinstance(res, (list,str)) or anno is empty: return res
    return [_fix_anno(anno, o) for o in res] if isinstance(res,list) else _fix_anno(anno, res)

_core._find_wsp = _find_wsp_patch

In [ ]:
# CHANGED: Added htmx4 parameter and pass it to _find_wsp
_find_wsp = _core._find_wsp
def _wrap_ws(ws, data, params, htmx4=False):
    hdrs = {k.lower().replace('-','_'):v for k,v in data.pop('HEADERS', {}).items()}
    return {arg:_find_wsp(ws, data, hdrs, arg, p, htmx4=htmx4) for arg,p in params.items()}
_core._wrap_ws = _wrap_ws

In [ ]:
# CHANGED: Added htmx4 param, pass to _wrap_ws, use _send_ws4 vs _send_ws based on htmx4
def _ws_endp(recv, conn=None, disconn=None, htmx4=False):
    cls = type('WS_Endp', (WebSocketEndpoint,), {"encoding":"text"})
    async def _generic_handler(handler, ws, data=None):
        wd = _wrap_ws(ws, loads(data) if data else {}, _params(handler), htmx4=htmx4)
        resp = await _handle(handler, **wd)
        if resp:
            if htmx4: await _send_ws4(ws, resp)
            else: await _send_ws(ws, resp)

    async def _connect(self, ws):
        await ws.accept()
        await _generic_handler(conn, ws)

    async def _disconnect(self, ws, close_code): await _generic_handler(disconn, ws)
    async def _recv(self, ws, data): await _generic_handler(recv, ws, data)

    if    conn: cls.on_connect    = _connect
    if disconn: cls.on_disconnect = _disconnect
    cls.on_receive = _recv
    return cls
_core._ws_endp = _ws_endp

## Test WS

### Test WS htmx2

### Test WS htmx4

In [ ]:
from fasthtml.jupyter import *

from asyncio import sleep

app = FastHTML(exts='ws', htmx=False, htmx4=True)
rt = app.route

def mk_inp(): return Input(id='msg', name='msg')
nid = 'notifications'

@rt('/')
async def get():
    cts = Div(
        Div(id=nid),
        Form(mk_inp(), id='form', hx_ws_send=True),
        hx_ws_connect='/ws')
    return Titled('Websocket Test', cts)

async def on_connect(send): 
    await send(WsResponse4(Div('Hello, you have connected', id=nid), target=f'#{nid}'))
async def on_disconnect(): print('Disconnected!')

@app.ws('/ws', conn=on_connect, disconn=on_disconnect)
async def ws(msg:str, send):

# Note: Can also use dict instead of WsResponse4, e.g.:
# await send({'content': Div('Hello', id=nid), 'target': f'#{nid}'})
    await send(WsResponse4(Div('Hello ' + msg, id=nid), target=f'#{nid}'))
    await sleep(2)
    await send(WsResponse4(Div('Goodbye ' + msg, id=nid), target=f'#{nid}'))
    await send(WsResponse4(mk_inp(), target='#msg'))

srv = JupyUvi(app)

Disconnected!
